# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get all record sets and their fields

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the Croissant metadata. This is likely a single-record-set dataset.")

# mlcroissant typically exposes record sets, fields, and columns via .record_sets, .fields, etc. as lists of objects
for rset in dataset.record_sets:
    print(f"Record set: {rset.name} (@id: {rset.id})")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

In [ ]:
# List record set @ids
record_set_ids = [rset.id for rset in dataset.record_sets]
print("All available Record Set @ids:", record_set_ids)

# For this dataset, we typically have one main survey record set.
# Let's pick the first (or only) record set.
main_record_set_id = record_set_ids[0]

# Load all records from the selected record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in main record set ({main_record_set_id}):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let us apply some typical data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attribute for further analysis.

We'll use the field `phq9_total_score` as an example numeric field and group by `gender` if present.

In [ ]:
# Identify numeric field and group field by @id
phq9_score_field = 'phq9_total_score'    # Example field from this mental health dataset
gender_field = 'gender'                 # Categorical field for grouping

# Filter: E.g. focus on participants with moderate or higher PHQ-9
threshold = 10  # Moderate depression cut-off for PHQ-9
if phq9_score_field in df.columns:
    filtered_df = df[df[phq9_score_field] > threshold]
    print(f"Filtered records with {phq9_score_field} > {threshold}:")
    print(filtered_df[[phq9_score_field, gender_field]].head())

    # Normalize the numeric field
    filtered_df[f"{phq9_score_field}_normalized"] = (
        filtered_df[phq9_score_field] - filtered_df[phq9_score_field].mean()
    ) / filtered_df[phq9_score_field].std()
    print(f"Normalized {phq9_score_field} for filtered records:")
    print(filtered_df[[phq9_score_field, f"{phq9_score_field}_normalized"]].head())

    # Group by gender
    if gender_field in filtered_df.columns:
        grouped = (
            filtered_df.groupby(gender_field)[phq9_score_field]
            .agg(['mean', 'std', 'count'])
            .reset_index()
        )
        print(f"Grouped data by {gender_field}:")
        print(grouped)
else:
    print(f"Field {phq9_score_field} not found in the dataset.")

## 5. Visualization
Let's visualize the distribution of PHQ-9 total scores and compare across genders.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
if phq9_score_field in df.columns and gender_field in df.columns:
    sns.histplot(
        data=df, x=phq9_score_field, hue=gender_field,
        bins=15, kde=True, palette="muted", multiple="stack"
    )
    plt.title("Distribution of PHQ-9 Total Score by Gender")
    plt.xlabel("PHQ-9 Total Score")
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Cannot plot: {phq9_score_field} and/or {gender_field} not found in columns:", df.columns.tolist())

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using Croissant metadata with the `mlcroissant` library. We examined the schema, extracted and filtered numeric data using record set and field `@id` references, and visualized key mental health indicators such as PHQ-9 scores. Further analyses can apply the same template to other fields such as GAD-7 or PSQ scores for deeper insight into mental health trends in this population.